# Run Inference with Hugging Face Models

This tutorial shows how to run inference across the main modalities supported by `Transformers.jl`:

- Text generation (autoregressive / chat)
- Embeddings and semantic similarity
- Text classification
- Image classification
- Speech recognition (ASR)
- Multimodal (CLIP image–text similarity)
- Batching for throughput
- GPU acceleration

Prerequisite: you have loaded a model and tokenizer as shown in **Tutorial 01 — Load a Model**.

In [ ]:
Pollis.packages("Transformers", "Flux", "Images", "WAV", "CUDA")

In [ ]:
using Transformers, Transformers.HuggingFace

## 1. Text Generation

Decoder-only models (GPT-2, Llama, Mistral, Gemma) generate text autoregressively. The `generate` function extends the input token sequence one token at a time until `max_new_tokens` is reached or an end-of-sequence token is produced.

In [ ]:
tkr   = HuggingFace.load_tokenizer("gpt2")
model = HuggingFace.load_model(HuggingFace.GPT2Model, "gpt2")

# Tokenise the prompt
enc = tkr("Julia is a high-performance programming language designed for")

# Generate — returns a vector of token-id vectors (one per sequence)
out  = generate(model, enc.token;
    max_new_tokens    = 40,
    temperature       = 0.8,
    top_p             = 0.95,
    repetition_penalty = 1.1,
)

# Decode the first (and only) output sequence
println(tkr.decode(out[1]))

## 2. Chat / Instruction Models

Instruct-tuned models expect a specific prompt template. The tokenizer's `apply_chat_template` method formats a list of role–content messages into the correct string.

In [ ]:
# Using Llama-3.2-3B-Instruct — set HF_TOKEN if not already set
# ENV["HF_TOKEN"] = "hf_..."

tkr_llama   = HuggingFace.load_tokenizer("meta-llama/Llama-3.2-3B-Instruct")
model_llama = HuggingFace.load_model(HuggingFace.LlamaModel, "meta-llama/Llama-3.2-3B-Instruct")

messages = [
    Dict("role" => "system",  "content" => "You are a concise Julia programming assistant."),
    Dict("role" => "user",    "content" => "What is multiple dispatch and why does Julia use it?"),
]

# Format with the model's chat template
prompt = tkr_llama.apply_chat_template(messages; tokenize=false, add_generation_prompt=true)
enc    = tkr_llama(prompt)

out  = generate(model_llama, enc.token;
    max_new_tokens = 256,
    temperature    = 0.7,
    top_p          = 0.9,
)
println(tkr_llama.decode(out[1]))

## 3. Embeddings and Semantic Similarity

Encoder models (BERT, RoBERTa, sentence-transformers) produce a fixed-size vector for each input. The CLS token embedding (first position) is the standard sentence representation.

Semantic similarity is measured with cosine similarity: two sentences that mean the same thing will have a score close to 1.

In [ ]:
using LinearAlgebra

tkr_emb   = HuggingFace.load_tokenizer("sentence-transformers/all-MiniLM-L6-v2")
model_emb = HuggingFace.load_model(HuggingFace.BertModel, "sentence-transformers/all-MiniLM-L6-v2")

function embed(text::String)
    enc = tkr_emb(text)
    out = model_emb(enc.token, enc.segment, enc.attention_mask)
    v   = vec(out.hidden_state[:, 1, :])   # CLS token
    v ./ norm(v)                            # L2-normalise
end

e1 = embed("Julia is a fast language for scientific computing")
e2 = embed("Julia has excellent performance for numerical work")
e3 = embed("The weather in Lisbon is warm and sunny")

println("Similar sentences:     ", round(dot(e1, e2), digits=4))
println("Dissimilar sentences:  ", round(dot(e1, e3), digits=4))

## 4. Text Classification

Classification models output a `logits` vector with one entry per class. Apply `softmax` to get probabilities and `argmax` to get the predicted class index.

In [ ]:
using Flux: softmax

# distilbert-base-uncased-finetuned-sst-2-english: positive/negative sentiment
tkr_cls   = HuggingFace.load_tokenizer("distilbert-base-uncased-finetuned-sst-2-english")
model_cls = HuggingFace.load_model(HuggingFace.BertForSequenceClassification,
                                    "distilbert-base-uncased-finetuned-sst-2-english")

for text in ["I absolutely love Julia!", "The package API is confusing and poorly documented."]
    enc      = tkr_cls(text)
    out      = model_cls(enc.token, enc.segment, enc.attention_mask)
    probs    = softmax(vec(out.logits))
    pred_idx = argmax(probs)
    label    = model_cls.config.id2label[pred_idx]
    println(rpad(text, 55), "  →  ", label, " (", round(100 * probs[pred_idx], digits=1), "%)")
end

## 5. Image Classification

Vision models use a *processor* to resize and normalise images before passing them to the model. The output `logits` vector maps to class labels via `id2label`.

In [ ]:
using Images

proc_vit  = HuggingFace.load_processor("google/vit-base-patch16-224")
model_vit = HuggingFace.load_model(HuggingFace.ViTForImageClassification,
                                    "google/vit-base-patch16-224")

# Load any JPEG/PNG image
img    = load("photo.jpg")
inputs = proc_vit(img)                         # resize to 224×224, normalise
out    = model_vit(inputs.pixel_values)

pred_idx   = argmax(vec(out.logits))
pred_label = model_vit.config.id2label[pred_idx]
println("Predicted class: ", pred_label)

## 6. Speech Recognition (ASR)

Whisper transcribes audio to text. Load the audio as a 16 kHz mono waveform, pass it through the processor to get a log-mel spectrogram, then call `generate`.

In [ ]:
using WAV

proc_whisper  = HuggingFace.load_processor("openai/whisper-large-v3")
model_whisper = HuggingFace.load_model(HuggingFace.WhisperForConditionalGeneration,
                                        "openai/whisper-large-v3")

# Load a 16 kHz mono WAV file
wave, sr = wavread("speech.wav")

# Processor converts waveform → log-mel spectrogram
inputs = proc_whisper(wave)

# Generate transcript
out  = generate(model_whisper, inputs.input_features)
text = proc_whisper.decode(out[1])
println("Transcript: ", text)

## 7. Multimodal — CLIP Image–Text Similarity

CLIP encodes images and texts into a shared embedding space. The `logits_per_image` output gives the similarity score between the image and each candidate text label.

In [ ]:
using Images, Flux: softmax

proc_clip  = HuggingFace.load_processor("openai/clip-vit-base-patch32")
model_clip = HuggingFace.load_model(HuggingFace.CLIPModel, "openai/clip-vit-base-patch32")

img   = load("dog.jpg")
labels = ["a dog", "a cat", "a car", "a bicycle"]

img_inputs = proc_clip(img)
txt_inputs = proc_clip(labels)

out   = model_clip(img_inputs.pixel_values, txt_inputs.input_ids, txt_inputs.attention_mask)
probs = softmax(vec(out.logits_per_image))

for (label, prob) in zip(labels, probs)
    println(rpad(label, 15), "  ", round(100 * prob, digits=1), "%")
end
println("Best match: ", labels[argmax(probs)])

## 8. Batching for Throughput

Processing one input at a time has high overhead. Pad inputs to the same length and process them as a batch — the model runs once and returns results for all inputs simultaneously.

In [ ]:
tkr_bert   = HuggingFace.load_tokenizer("bert-base-uncased")
model_bert = HuggingFace.load_model(HuggingFace.BertModel, "bert-base-uncased")

texts = [
    "Julia is a high-level, high-performance language.",
    "Transformers.jl brings the Hugging Face ecosystem to Julia.",
    "The Hugging Face Hub hosts over 500,000 pretrained models.",
    "Pollis integrates AI tools into the development workflow.",
]

# Encode all texts at once — padding pads shorter sequences to the longest
enc = tkr_bert(texts; padding=true, truncation=true, max_length=128)

# Single forward pass for the entire batch
out = model_bert(enc.token, enc.segment, enc.attention_mask)

# out.hidden_state shape: (hidden_dim, seq_len, batch_size)
println("Output shape: ", size(out.hidden_state))

# Extract CLS embeddings for all sentences
cls_embeddings = out.hidden_state[:, 1, :]   # (hidden_dim, batch_size)
println("CLS embeddings shape: ", size(cls_embeddings))

## 9. GPU Acceleration

Moving the model and inputs to GPU with `Flux.gpu` typically gives 5–20× speedup for medium and large models. Move the model once at startup; move inputs at inference time.

In [ ]:
using CUDA, Flux

# Check GPU availability
if CUDA.functional()
    println("GPU: ", CUDA.name(CUDA.device()))
else
    println("No CUDA GPU detected — running on CPU")
end

# Move model to GPU once at startup (no-op on CPU if CUDA not available)
model_gpu = model_bert |> gpu

function embed_gpu(text::String)
    enc   = tkr_bert(text)
    token = enc.token |> gpu
    seg   = enc.segment |> gpu
    mask  = enc.attention_mask |> gpu
    out   = model_gpu(token, seg, mask)
    vec(out.hidden_state[:, 1, :]) |> cpu
end

v = embed_gpu("Julia on GPU is very fast")
println("Embedding dim: ", length(v))

## 10. Encoder-Decoder Models (T5)

Seq2seq models such as T5 and FLAN-T5 use `generate` with an additional `decoder_start_token_id` argument. They excel at translation, summarisation, and question answering.

In [ ]:
tkr_t5   = HuggingFace.load_tokenizer("google/flan-t5-base")
model_t5 = HuggingFace.load_model(HuggingFace.T5Model, "google/flan-t5-base")

# T5-style task prefix
enc = tkr_t5("translate English to French: Machine learning is transforming science.")

out  = generate(model_t5, enc.token;
    decoder_start_token_id = model_t5.config.decoder_start_token_id,
    max_new_tokens         = 64,
)
println(tkr_t5.decode(out[1]))

## Summary

| Task | Pattern |
|---|---|
| Text generation | `generate(model, enc.token; max_new_tokens=N)` |
| Chat/instruct | `tkr.apply_chat_template(messages)` then `generate` |
| Embeddings | `model(enc.token, enc.segment, enc.attention_mask).hidden_state[:, 1, :]` |
| Classification | `softmax(model(...).logits)` + `id2label` |
| Image classification | `proc(img)` then `model(inputs.pixel_values)` |
| ASR | `proc(wave)` then `generate(model, inputs.input_features)` |
| CLIP similarity | `model(img_px, txt_ids, txt_mask).logits_per_image` |
| Batching | pass `Vector{String}` to tokenizer with `padding=true` |
| GPU | `model \|> gpu`, `enc.token \|> gpu` |